# TS convergence, one structure: dependency chain

`refine_temperature_bracket_with_chain`: submit once, then wait or walk away.

The whole candidate sequence of brackets is decided **up front** and every bracket is submitted at once. The first has no dependency; each later one gets the previous one's `Future` as an argument. executorlib sees the `Future` and holds the task back until it is done:

* `SingleNodeExecutor`: a thread in your Python process does it, so the process has to stay alive.
* `SlurmClusterExecutor` / `FluxClusterExecutor`: the dependency is passed on to the scheduler (`sbatch --dependency=afterok:<jobid>`), so nothing of yours has to keep running.

Consequences: the brackets are fixed before any result exists (a step is not chosen from the criterion of the previous one), and jobs are reserved for brackets that may not be needed; a step whose predecessor already converged just returns that result. If the bracket log already holds a bracket that satisfies the tolerance, nothing is submitted at all.

In [1]:
import sys
from pathlib import Path

# _common.py (shared settings) sits two levels up from this notebook
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "_common.py").is_file())))
import _common

_common.setup_environment()


In [2]:
import os

import pandas as pd
from executorlib import SingleNodeExecutor

from phase_diagram_workflows.free_energies.ts_convergence.single import (
    load_bracket_history,
    refine_temperature_bracket_with_chain,
)

## 1. Inputs

In [3]:
structure = _common.al_structure()                 # 32 atoms
potential_df = _common.mishin_al_potential()
calphy_parameters = _common.calphy_parameters()
INITIAL_BRACKET = _common.INITIAL_BRACKET          # (700, 720) K
print(f"{len(structure)} atoms, bracket {INITIAL_BRACKET}")


32 atoms, bracket (700.0, 720.0)


## 2. Submit the chain

`refine_temperature_bracket_with_chain` never builds an executor: it only calls `.submit()` on the one you hand it. The small wrapper below counts those calls, to show that every bracket really goes through your object (a `SlurmClusterExecutor` would work the same).

`tolerance=-1` is unreachable, so each bracket is narrowed: `step_upper=5` and `max_iterations=2` give `(700, 720)` -> `(700, 715)` -> `(700, 710)`.

In [4]:
class SubmitCountingExecutor:
    def __init__(self, wrapped):
        self._wrapped = wrapped
        self.submit_count = 0

    def submit(self, fn, **kwargs):
        self.submit_count += 1
        return self._wrapped.submit(fn, **kwargs)

    def shutdown(self, *args, **kwargs):
        return self._wrapped.shutdown(*args, **kwargs)


root = str(_common.results_directory("single_chain"))
real_executor = SingleNodeExecutor(hostname_localhost=True, max_cores=1, cache_directory=os.path.join(root, "executorlib_cache"))
executor = SubmitCountingExecutor(real_executor)

futures, first_working_directory = refine_temperature_bracket_with_chain(
    input_structure=structure,
    calphy_parameters=calphy_parameters,
    potential_df=potential_df,
    executor=executor,
    working_directory_root=root,
    initial_bracket=INITIAL_BRACKET,
    tolerance=-1.0,
    step_upper=5.0,
    max_iterations=2,
)
print(f"{len(futures)} brackets submitted up front, {executor.submit_count} submit() calls on our executor")

3 brackets submitted up front, 3 submit() calls on our executor


## 3. Wait for the last one

Waiting on the **last** future only works if every earlier step was driven to completion by the dependency resolution.

In [5]:
last = futures[-1].result()
real_executor.shutdown(wait=True)
print("last bracket:", last["bracket"], "| status:", last["status"])
assert executor.submit_count == len(futures) == 3

Input validation successful. Proceeding with calculation.


Input validation successful. Proceeding with calculation.


Input validation successful. Proceeding with calculation.


last bracket: (700.0, 710.0) | status: narrowed


In [6]:
pd.read_csv(os.path.join(root, "bracket_log.csv"))

,t_low,t_high,criterion
0,700.0,720.0,0.005836
1,700.0,715.0,0.010062
2,700.0,710.0,0.003978


## 4. A looser tolerance later recovers a wider bracket, with no new calculation

Everything that was tried is in the log. Asking again with a tolerance that one of the finished brackets satisfies submits nothing: you get an already-resolved future for the widest bracket that is good enough.

In [7]:
log = pd.read_csv(os.path.join(root, "bracket_log.csv"))
loose = float(log["criterion"].max()) * 1.01          # every finished bracket now counts as good
before = sorted(load_bracket_history(root)[0])

executor = SubmitCountingExecutor(SingleNodeExecutor(hostname_localhost=True, max_cores=1, cache_directory=os.path.join(root, "executorlib_cache")))
futures, _ = refine_temperature_bracket_with_chain(
    input_structure=structure, calphy_parameters=calphy_parameters, potential_df=potential_df,
    executor=executor, working_directory_root=root, initial_bracket=INITIAL_BRACKET,
    tolerance=loose, step_upper=5.0, max_iterations=2,
)
result = futures[0].result()
print("tolerance", round(loose, 5), "->", result["status"], result["bracket"], "| submit() calls:", executor.submit_count)
assert result["status"] == "converged" and executor.submit_count == 0 and sorted(load_bracket_history(root)[0]) == before

tolerance 0.01016 -> converged (700.0, 720.0) | submit() calls: 0
